# Improved Patient Booking System

This is an enhanced version with:
- Input validation and error handling
- Better code organization with functions
- Menu-driven interface
- Data persistence (loads existing records)
- Search and update capabilities
- Type hints and documentation


In [ ]:
import csv
import os
from datetime import datetime
from typing import List, Optional
import pandas as pd
import re

In [ ]:
class Patient:
    """Represents a patient with validation and data conversion methods."""
    
    def __init__(self, patient_id: str, name: str, age: int, gender: str, 
                 phone: str, pain_level: str, report_time: str, 
                 first_visit: str, complaint: str, address: str, 
                 emergency_contact: str):
        """
        Initialize a patient record.
        
        Args:
            patient_id: Unique identifier for the patient
            name: Patient's full name
            age: Patient's age (0-150)
            gender: Patient's gender
            phone: Patient's phone number
            pain_level: Level of pain (MILD, MODERATE, SEVERE)
            report_time: Time of report
            first_visit: Whether this is first visit (Yes/No)
            complaint: Main complaint/reason for visit
            address: Patient's address
            emergency_contact: Emergency contact information
        """
        self.patient_id = patient_id
        self.name = name
        self.age = age
        self.gender = gender
        self.phone = phone
        self.pain_level = pain_level
        self.report_time = report_time
        self.first_visit = first_visit
        self.complaint = complaint
        self.address = address
        self.emergency_contact = emergency_contact
        self.created_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    def to_dict(self) -> dict:
        """Convert patient object to dictionary for CSV writing."""
        return {
            "Patient ID": self.patient_id,
            "Name": self.name,
            "Age": self.age,
            "Gender": self.gender,
            "Phone": self.phone,
            "Pain Level": self.pain_level,
            "Report Time": self.report_time,
            "First Visit": self.first_visit,
            "Complaint": self.complaint,
            "Address": self.address,
            "Emergency Contact": self.emergency_contact,
            "Created At": self.created_at
        }
    
    def __str__(self) -> str:
        """String representation of patient."""
        return f"Patient(ID: {self.patient_id}, Name: {self.name}, Age: {self.age})"

print("✓ Patient class defined")

In [ ]:
class InputValidator:
    """Validates user input for patient data."""
    
    PAIN_LEVELS = ["MILD", "MODERATE", "SEVERE"]
    GENDERS = ["MALE", "FEMALE", "OTHER"]
    
    @staticmethod
    def validate_age(age: int) -> bool:
        """Validate age is between 0 and 150."""
        return 0 <= age <= 150
    
    @staticmethod
    def validate_phone(phone: str) -> bool:
        """Validate phone number (at least 10 digits)."""
        return bool(re.match(r'^[\d\+\-\s()]{10,}$', phone))
    
    @staticmethod
    def validate_pain_level(pain_level: str) -> bool:
        """Validate pain level is from predefined list."""
        return pain_level.upper() in InputValidator.PAIN_LEVELS
    
    @staticmethod
    def validate_yes_no(value: str) -> bool:
        """Validate yes/no input."""
        return value.lower() in ['yes', 'no', 'y', 'n']

print("✓ InputValidator class defined")

In [ ]:
class PatientManager:
    """Manages patient records: storage, retrieval, and persistence."""
    
    def __init__(self, filename: str = "patients.csv"):
        """Initialize the patient manager with a CSV file."""
        self.filename = filename
        self.patients: List[Patient] = []
        self.load_patients()
    
    def load_patients(self) -> None:
        """Load existing patients from CSV file."""
        if os.path.exists(self.filename):
            try:
                df = pd.read_csv(self.filename)
                for _, row in df.iterrows():
                    patient = Patient(
                        patient_id=str(row["Patient ID"]),
                        name=str(row["Name"]),
                        age=int(row["Age"]),
                        gender=str(row["Gender"]),
                        phone=str(row["Phone"]),
                        pain_level=str(row["Pain Level"]),
                        report_time=str(row["Report Time"]),
                        first_visit=str(row["First Visit"]),
                        complaint=str(row["Complaint"]),
                        address=str(row["Address"]),
                        emergency_contact=str(row["Emergency Contact"])
                    )
                    self.patients.append(patient)
                print(f"✓ Loaded {len(self.patients)} existing patient(s)")
            except Exception as e:
                print(f"⚠ Error loading patients: {e}")
    
    def add_patient(self, patient: Patient) -> None:
        """Add a new patient to the list."""
        if self.get_patient_by_id(patient.patient_id):
            print(f"⚠ Patient with ID {patient.patient_id} already exists!")
            return
        self.patients.append(patient)
        print(f"✓ Patient {patient.name} added successfully")
    
    def get_patient_by_id(self, patient_id: str) -> Optional[Patient]:
        """Search for a patient by ID."""
        for patient in self.patients:
            if patient.patient_id == patient_id:
                return patient
        return None
    
    def search_by_name(self, name: str) -> List[Patient]:
        """Search for patients by name (case-insensitive)."""
        return [p for p in self.patients if name.lower() in p.name.lower()]
    
    def get_all_patients(self) -> List[Patient]:
        """Return all patients."""
        return self.patients
    
    def save_patients(self) -> None:
        """Save all patients to CSV file."""
        if not self.patients:
            print("⚠ No patients to save")
            return
        
        try:
            with open(self.filename, "w", newline="") as file:
                fieldnames = [
                    "Patient ID", "Name", "Age", "Gender", "Phone",
                    "Pain Level", "Report Time", "First Visit", "Complaint",
                    "Address", "Emergency Contact", "Created At"
                ]
                writer = csv.DictWriter(file, fieldnames=fieldnames)
                writer.writeheader()
                for patient in self.patients:
                    writer.writerow(patient.to_dict())
            print(f"✓ {len(self.patients)} patient(s) successfully saved to '{self.filename}'")
        except Exception as e:
            print(f"✗ Error saving patients: {e}")
    
    def delete_patient(self, patient_id: str) -> bool:
        """Delete a patient by ID."""
        for i, patient in enumerate(self.patients):
            if patient.patient_id == patient_id:
                self.patients.pop(i)
                print(f"✓ Patient {patient_id} deleted successfully")
                return True
        print(f"✗ Patient {patient_id} not found")
        return False

print("✓ PatientManager class defined")

In [ ]:
def get_valid_age() -> int:
    """Get and validate age input from user."""
    while True:
        try:
            age = int(input("Age (0-150): "))
            if InputValidator.validate_age(age):
                return age
            else:
                print("✗ Age must be between 0 and 150")
        except ValueError:
            print("✗ Please enter a valid number")

def get_valid_phone() -> str:
    """Get and validate phone number from user."""
    while True:
        phone = input("Phone Number: ")
        if InputValidator.validate_phone(phone):
            return phone
        else:
            print("✗ Please enter a valid phone number (at least 10 digits)")

def get_valid_pain_level() -> str:
    """Get and validate pain level from user."""
    while True:
        pain = input(f"Pain Level {InputValidator.PAIN_LEVELS}: ").upper()
        if InputValidator.validate_pain_level(pain):
            return pain
        else:
            print(f"✗ Please choose from {InputValidator.PAIN_LEVELS}")

def get_valid_yes_no(prompt: str) -> str:
    """Get and validate yes/no input from user."""
    while True:
        response = input(prompt).lower()
        if InputValidator.validate_yes_no(response):
            return "Yes" if response in ['yes', 'y'] else "No"
        else:
            print("✗ Please enter 'yes' or 'no'")

print("✓ Input validation functions defined")

In [ ]:
def collect_patient_data() -> Patient:
    """Collect and validate patient information from user input."""
    print("\n" + "="*50)
    print("ENTER PATIENT DETAILS")
    print("="*50)
    
    patient_id = input("Patient ID: ").strip()
    name = input("Name: ").strip()
    age = get_valid_age()
    gender = input("Gender (Male/Female/Other): ").strip()
    phone = get_valid_phone()
    pain_level = get_valid_pain_level()
    report_time = input("Report Time (HH:MM): ").strip()
    first_visit = get_valid_yes_no("First Visit? (Yes/No): ")
    complaint = input("Complaint/Reason for Visit: ").strip()
    address = input("Address: ").strip()
    emergency_contact = input("Emergency Contact: ").strip()
    
    return Patient(
        patient_id, name, age, gender, phone,
        pain_level, report_time, first_visit,
        complaint, address, emergency_contact
    )

print("✓ Patient input function defined")

In [ ]:
def display_patients(patients: List[Patient]) -> None:
    """Display patients in a formatted table."""
    if not patients:
        print("⚠ No patients to display")
        return
    
    df = pd.DataFrame([p.to_dict() for p in patients])
    print("\n" + "="*150)
    print(df.to_string(index=False))
    print("="*150)

def display_single_patient(patient: Patient) -> None:
    """Display a single patient's details."""
    print("\n" + "-"*40)
    for key, value in patient.to_dict().items():
        print(f"{key:.<30} {value}")
    print("-"*40)

print("✓ Display functions defined")

In [ ]:
def display_menu() -> None:
    """Display the main menu."""
    print("\n" + "="*50)
    print("PATIENT BOOKING SYSTEM")
    print("="*50)
    print("1. Add New Patient")
    print("2. View All Patients")
    print("3. Search Patient by ID")
    print("4. Search Patient by Name")
    print("5. Delete Patient")
    print("6. Save & Export Data")
    print("7. Exit")
    print("="*50)

def run_application() -> None:
    """Run the patient booking system application."""
    manager = PatientManager()
    
    while True:
        display_menu()
        choice = input("Select an option (1-7): ").strip()
        
        if choice == "1":
            try:
                patient = collect_patient_data()
                manager.add_patient(patient)
            except Exception as e:
                print(f"✗ Error adding patient: {e}")
        
        elif choice == "2":
            patients = manager.get_all_patients()
            display_patients(patients)
        
        elif choice == "3":
            patient_id = input("Enter Patient ID: ").strip()
            patient = manager.get_patient_by_id(patient_id)
            if patient:
                display_single_patient(patient)
            else:
                print(f"✗ Patient {patient_id} not found")
        
        elif choice == "4":
            name = input("Enter Patient Name (partial search): ").strip()
            results = manager.search_by_name(name)
            display_patients(results)
        
        elif choice == "5":
            patient_id = input("Enter Patient ID to delete: ").strip()
            confirm = input(f"Are you sure? (yes/no): ").lower()
            if confirm in ['yes', 'y']:
                manager.delete_patient(patient_id)
            else:
                print("✗ Deletion cancelled")
        
        elif choice == "6":
            manager.save_patients()
        
        elif choice == "7":
            confirm = input("Save before exiting? (yes/no): ").lower()
            if confirm in ['yes', 'y']:
                manager.save_patients()
            print("\n✓ Thank you for using Patient Booking System. Goodbye!")
            break
        
        else:
            print("✗ Invalid option. Please select 1-7")

print("✓ Application functions defined")

In [ ]:
# Run the application
if __name__ == "__main__":
    run_application()

## Key Improvements Made

### 1. **Code Organization**
- Separated concerns into dedicated classes: `Patient`, `InputValidator`, `PatientManager`
- Each function has a single responsibility
- Added docstrings and type hints throughout

### 2. **Input Validation**
- Age validation (0-150 range)
- Phone number validation (regex pattern)
- Pain level validation (predefined options)
- Yes/No input validation
- Continuous prompting until valid input received

### 3. **Data Persistence**
- Loads existing patient records on startup
- Appends new records instead of overwriting
- Added timestamp tracking (Created At field)
- Handles file errors gracefully

### 4. **Menu-Driven Interface**
- User-friendly menu with 7 options
- Add, view, search, and delete patients
- Save data at any time
- Confirmation prompts for destructive actions

### 5. **Search Capabilities**
- Search by Patient ID (exact match)
- Search by Patient Name (partial/case-insensitive)
- Display results in formatted tables

### 6. **Better Error Handling**
- Try-except blocks for file operations
- Validation loops prevent invalid data
- User-friendly error messages with indicators (✓, ✗, ⚠)
- Duplicate ID prevention

### 7. **User Experience**
- Clear visual separation with borders and headers
- Status indicators for success/failure
- Formatted table display using pandas
- Confirmation prompts before critical actions
